# Day 1 · NumPy

**Block:** NumPy (80 min, including slides and live coding)

The block alternates slides, live coding and exercises. Each part below
matches one "Your turn" slide:

1. **Indexing** (12 min): time windows from chosen channels, and trials from
   3-D spike counts
2. **Aggregations and broadcasting** (15 min): summaries along axes,
   re-referencing and baseline correction
3. **Masks and vectorising** (18 min): select trials and channels by
   condition, time a loop against NumPy, then the mouse's psychometric curve
   without a single loop
4. **Check your work with a plot** (5 min)
5. **Stretch**, for anyone who finishes early: views and copies, in-place
   changes, dtypes, more broadcasting, fancy indexing and data statistics

The slides and live coding used 2-D tables of scores and a week of
temperatures. The exercises use one real experiment, a recording from a
mouse brain described below; only a few small SPL exercises use made-up
arrays.

Most tasks ask you to store a result in a named variable. Print it, and
check its `.shape` and values against the array you started from. Part 4
shows just enough Matplotlib to check a result by eye; plotting proper is on
Day 2.

The SPL exercises are adapted from [Scientific Python Lectures, NumPy
exercises](https://lectures.scientific-python.org/intro/numpy/exercises.html)
(CC BY 4.0). The rest covers the same ground as the Python Data Science
Handbook chapters on [array
basics](https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html),
[broadcasting](https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html),
[masks](https://jakevdp.github.io/PythonDataScienceHandbook/02.06-boolean-arrays-and-masks.html)
and [fancy
indexing](https://jakevdp.github.io/PythonDataScienceHandbook/02.07-fancy-indexing.html).
Read those for the full explanations.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(seed=0)

## The recording

A mouse sits in front of a screen. A striped patch (the stimulus) appears on
the left or the right, faint or strong, and the mouse turns a wheel to move it
to the centre. A correct turn earns a drop of water. Meanwhile a
**Neuropixels probe** records from the mouse's brain.

The data come from the [International Brain Laboratory](https://www.internationalbrainlab.com/)
and were downloaded from the [DANDI archive](https://dandiarchive.org/dandiset/000409)
(CC BY 4.0; [doi:10.48324/dandi.000409/0.260309.1324](https://doi.org/10.48324/dandi.000409/0.260309.1324)).
`../data/ibl_session.h5` holds a small slice of one session. It is
an HDF5 file: a file of named arrays, arranged in groups like folders. The
`h5py` package reads it, and `[:]` turns each one into a NumPy array:

In [ ]:
with h5py.File("../data/ibl_session.h5") as f:
    # LFP: the voltage at every site, 500 times a second, for 10 s
    lfp = f["lfp/data"][:]
    uV_per_count = f["lfp"].attrs["uV_per_count"]
    sampling_rate = f["lfp"].attrs["sampling_rate_Hz"]
    lfp_start = f["lfp"].attrs["start_time_s"]
    # One entry per recording site (channel)
    area = f["electrodes/area"][:].astype(str)
    layer = f["electrodes/layer"][:].astype(str)
    x_um = f["electrodes/x_um"][:]
    depth_um = f["electrodes/depth_um"][:]
    # One entry per trial
    stim_on = f["trials/stim_on_s"][:]
    contrast = f["trials/contrast"][:]
    probability_left = f["trials/probability_left"][:]
    choice = f["trials/choice"][:]
    rt = f["trials/response_time_s"][:]
    correct = f["trials/correct"][:]
    # Spike counts of 51 neurons, in 50 ms bins around each stimulus
    spike_counts = f["spike_counts/data"][:]
    bin_start = f["spike_counts/bin_start_s"][:]
    # One entry per neuron (unit), including those not in spike_counts
    unit_channel = f["units/channel"][:]

# The raw LFP is in amplifier counts; this is the same data in microvolts
lfp_uv = lfp * uV_per_count

print(lfp.shape, lfp.dtype)
print(spike_counts.shape, spike_counts.dtype)
print(stim_on.shape, area.shape)

* **`lfp[time, channel]`** is the *local field potential*: the slow voltage
  changes near each site, from 618 s into the session (`lfp_start`). Row `i` is
  time `i / sampling_rate` seconds into the slice. The values are raw counts
  from the amplifier; **`lfp_uv`** is the same in microvolts.
* **Channels** are numbered from the **tip** of the probe: channel 0 is the
  deepest site, and `depth_um` gives each site's height above the tip. `area`
  says which brain area each site sits in (for example `SSp-n`, primary
  somatosensory cortex, nose), and the top few sites (`void`) are above the
  brain.
* **`spike_counts[unit, trial, bin]`** is the number of spikes neuron `unit`
  fired in time bin `bin` of trial `trial`. Bin `b` starts `bin_start[b]`
  seconds after the stimulus, from -0.5 s (before it) to 0.95 s.
* The **trial** arrays have one entry for each of the 533 trials: when the
  stimulus appeared (`stim_on`, seconds into the session), its `contrast` (in
  %, negative on the left, positive on the right), the block's
  `probability_left` (explained in part 3), the mouse's `choice` (-1 left, 1
  right), its response time `rt`, and whether it was `correct`.

In [ ]:
print(area[:12])
print(depth_um[:12])

## 1. Indexing

Using **one indexing expression each** (no loops, no typing values in),
make:

1. `deepest_trace`: the whole trace of channel 0, shape (5000,)
2. `one_value`: channel 100 at sample 1000. It should be 24.
3. `first_second`: every channel during the first second (500 samples),
   shape (500, 384)
4. `sites_50_to_200`: channels 50 up to (not including) 200, from 2 s to 4 s,
   shape (1000, 150)
5. `last_second`: every channel during the last second, shape (500, 384)
6. `every_other_site`: every other channel (0, 2, 4, ...), shape (5000, 192)
7. `top_first`: the whole recording with the channels in reverse order, so
   that column 0 is the **top** of the probe
8. `downsampled`: every 5th sample of every channel (100 samples per second),
   shape (1000, 384)

Check the `.shape` of each result against the one given.

In [ ]:
...  # your code here

In [ ]:
# Your code here


Now in `spike_counts`, which has three axes `[unit, trial, bin]`:

9. `one_unit_one_trial`: unit 3's counts in every bin of trial 10, shape (30,)
10. `after_stimulus`: every unit and trial, but only the bins from the
    stimulus on (`bin_start` 0 and later, which is bin 10 onwards), shape
    (51, 533, 20)
11. `first_bin`: the first bin of every trial for every unit, shape (51, 533)

Finally, a time window that you have to **compute** rather than read off:

12. Trial 89's stimulus appeared at `stim_on[89]` seconds into the session,
    which is somewhere in our LFP slice. Make `around_stimulus`: every channel
    from 0.5 s before to 1 s after it, shape (750, 384). (Hint: work out the
    sample number of the stimulus from `lfp_start` and `sampling_rate`, and
    turn it into an `int`.)

In [ ]:
# Your code here


## 2. Aggregations and broadcasting

### 2a. Aggregations along more than one axis

Back to `spike_counts[unit, trial, bin]`, shape (51, 533, 30). Make:

1. `total_per_unit`: the total number of spikes from each unit, over all
   trials and bins (51 values). (Hint: `axis` also accepts a tuple of axes.)
2. `psth`: for each unit, the mean count in each time bin, averaged over
   trials. Shape (51, 30). This is called a *peri-stimulus time histogram*.
3. `busiest_trial`: for each unit, the **number** of the trial in which it
   fired the most spikes (51 values).
4. `overall_mean`: the mean count over the whole array (one number)
5. `population`: the mean over units **and** trials, one value per bin, in
   spikes per second (divide by the 0.05 s bin width). Does the population
   fire more after the stimulus (bin 10 onwards) than before?

In [ ]:
# Your code here


*Your answer here.*

### 2b. Broadcasting

No loops in this section.

1. Noise that reaches every site at once (from the mouse moving, say) can be
   removed by subtracting, at each time point, the median over all channels.
   Make `lfp_car` (for *common average reference*) from `lfp_uv`, using
   `np.median` with `keepdims=True`. Its shape should still be (5000, 384).
2. Make `psth_change`: each unit's `psth` minus that unit's mean over the
   bins **before** the stimulus (the first 10 bins). Shape (51, 30); the first
   10 values of each row should average 0.

In [ ]:
# Your code here


3. The cell below should keep the spike counts of the **correct** trials and
   set the others to 0, by multiplying by `correct` (one `True`/`False` per
   trial), but it fails. Fix it.

In [ ]:
spike_counts * correct

In [ ]:
# Your code here


## 3. Masks and vectorising

### 3a. Boolean masks

`rt` holds the mouse's response time on each trial, in seconds from the
stimulus to its choice, and `correct` says whether the choice was right.
Some trials are odd: on a few the mouse was already turning the wheel, and on
others it lost interest.

In [ ]:
rt[:10].round(3), correct[:10]

Make:

1. `too_fast`: the response times below 0.15 s (faster than the mouse could
   have seen the stimulus)
2. `n_correct`: the number of correct trials
3. `mean_rt_correct`: the mean response time of the **correct** trials only
4. `slow_errors`: the response times of trials that were **wrong** and slower
   than 1 s
5. `rt_clean`: a **new** array in which every response time below 0.15 or
   above 5 s is replaced by `np.nan`. `rt` must not change. Then compare
   `rt.mean()` with `np.nanmean(rt_clean)`.

In [ ]:
# Your code here


Masks work on channels too. `area` has one entry per channel, so a mask built
from it picks out **columns** of `lfp_uv`. Make:

6. `ssp_lfp`: the LFP of every site in primary somatosensory cortex
   (`"SSp-n"`), shape (5000, 128)
7. `in_brain`: the LFP of every site **except** those above the brain
   (`"void"`), shape (5000, 378)

In [ ]:
# Your code here


### 3b. Vectorised versus loops

The **line length** of a signal is the sum of the absolute differences between
consecutive samples. It is large when the signal is busy, and is a cheap
measure of activity in seizure detection.

`line_length_loop(signals)` below computes it with `for` loops over the
channels and the samples. Write `line_length_numpy(signals)`, which does the
same with no loop at all. Each takes an array of shape (time, channel) and
returns one value per channel. Check that they agree on `first_2_s`, then time
both with `%timeit`. How many times faster is NumPy? (Hint for the NumPy version: `signals[1:] -
signals[:-1]` gives every difference at once, or use `np.diff`. See also the
Handbook's [Profiling and Timing
Code](https://jakevdp.github.io/PythonDataScienceHandbook/01.07-timing-and-profiling.html).)

In [ ]:
first_2_s = lfp_uv[:1000]  # the first 2 s, in microvolts


def line_length_loop(signals):
    n_time, n_channels = signals.shape
    result = np.zeros(n_channels)
    for channel in range(n_channels):
        total = 0.0
        for t in range(1, n_time):
            total += abs(signals[t, channel] - signals[t - 1, channel])
        result[channel] = total
    return result


def line_length_numpy(signals):
    ...  # your code here

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

### 3c. The psychometric curve, without a loop

Does the mouse do the task? The **psychometric curve** is the fraction of
trials on which it chose **right**, at each contrast. There are 9 contrasts:

In [ ]:
levels = np.unique(contrast)
levels

The obvious way is a loop over `levels`, picking out the trials at each one.
Do it with a mask and broadcasting instead. `y` is `True` where the mouse
chose right:

In [ ]:
y = choice == 1


1. `at_level`: a (533, 9) boolean array, `True` where trial `i` had contrast
   `levels[j]`. (Hint: compare `contrast` with `levels`. Which of them needs
   a new axis so that the result is (533, 9)?)
2. `n_at_level`: the number of trials at each contrast (9 values).
3. `frac_right`: the fraction of rightward choices at each contrast. (Hint:
   `at_level & y[:, np.newaxis]` is `True` where a trial was at that contrast
   **and** the mouse chose right. Count, then divide.)

The task also has **blocks** of trials. The session starts with 90 unbiased
trials (`probability_left` 0.5). Then, for a while, the stimulus appears on the
left 80% of the time (0.8), then on the right 80% of the time (0.2), and so on.

4. Make `frac_right_rb` and `frac_right_lb`: the same curve for the
   right-block trials (`probability_left == 0.2`) and the left-block trials
   (0.8) only. (Hint: select the rows of `at_level` and `y` with a mask
   first.) Compare the two at contrast 0, where there is nothing to see. Has
   the mouse learned the blocks?

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

## 4. Check your work with a plot

A column of numbers is hard to judge; a plot is easy. Here is all the
Matplotlib you need today. Day 2 covers it properly.

In [ ]:
time = np.arange(500) / sampling_rate  # the first second, in seconds

fig, ax = plt.subplots()
ax.plot(time, lfp_uv[:500, 100], label="channel 100")
ax.plot(time, lfp_uv[:500, 300], label="channel 300")
ax.set_xlabel("time (s)")
ax.set_ylabel("LFP (µV)")
ax.legend();

* `fig, ax = plt.subplots()` makes a figure with one set of axes
* `ax.plot(x, y)` draws a line; `ax.plot(x, y, "o-")` adds a marker at each
  point
* `ax.set_xlabel`, `ax.set_ylabel` and `ax.legend()` (which uses each line's
  `label=`) label it

Plot `frac_right_rb` and `frac_right_lb` against `levels` on one set of axes,
with markers, labelled axes and a legend. Does the plot agree with what you
concluded from the numbers?

In [ ]:
# Your code here


## 5. Stretch

For anyone who finishes early, in any order.

### 5a. Build an array without typing it in (SPL Exercise 22, part 1)

Form the 2-D array below **without typing it in explicitly**, and call it
`a`:

```
[[1,  6, 11],
 [2,  7, 12],
 [3,  8, 13],
 [4,  9, 14],
 [5, 10, 15]]
```

Then make `rows_2_and_4`, a new array containing its 2nd and 4th rows.

In [ ]:
# Your code here


### 5b. Views and copies

1. Make `channel_100`, the trace of channel 100. Its first 0.1 s (50 samples)
   is an artefact: set them to 0 **in `channel_100`**. Then print
   `lfp[:55, 100]`: did `lfp` change?
2. Reset `lfp` by reading it from the file again (the cell below). Now do the
   same again, but this time **without changing `lfp`**.

In [ ]:
# Your code here


In [ ]:
with h5py.File("../data/ibl_session.h5") as f:
    lfp = f["lfp/data"][:]

In [ ]:
# Your code here


3. Which of these share memory with `lfp`, and which are copies? Find out
   with `np.shares_memory(lfp, ...)` rather than guessing: `lfp[:, 100]`,
   `lfp[:, ::2]`, `lfp[:, [50, 200]]`, `lfp[lfp > 100]`, `lfp.T`,
   `lfp.ravel()`, `lfp.flatten()`.

In [ ]:
# Your code here


*Your answer here.*

### 5c. In place, or a new array?

A colleague wrote this function to shift a signal so that its minimum is 0.
`readings` are four LFP samples, in microvolts:

In [ ]:
def remove_offset(signal):
    signal -= signal.min()
    return signal


readings = np.array([12.0, 15.5, 13.2, 18.1])
shifted = remove_offset(readings)
print(shifted)
print(readings)

1. `readings` changed, although the function returned a result. Why?
2. Write `remove_offset_safe`, which returns the shifted signal but leaves its
   input alone. Check it on a fresh `readings`.

*Your answer here.*

In [ ]:
def remove_offset_safe(signal):
    ...  # your code here


readings = np.array([12.0, 15.5, 13.2, 18.1])

In [ ]:
# Your code here


NumPy's random generator has the same split. `rng.permutation(x)` returns a
shuffled copy and `rng.shuffle(x)` shuffles `x` itself. Shuffling trial
numbers is how you build a *shuffle control*: is a neuron's response tied to
the stimulus, or would any trial order do?

3. Make `shuffled_order`, a shuffled copy of `order`, leaving `order`
   unchanged. Print `order` to check.
4. Now shuffle `order` itself, in place. What does `rng.shuffle` return?

In [ ]:
order = np.arange(10)  # trial numbers

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

### 5d. dtypes

`lfp` holds whole numbers from the amplifier, stored as `int16` (2 bytes
each, from -32768 to 32767) to save space. That is why the first cell of this
notebook made `lfp_uv` as a **new** array: converting `lfp` to microvolts in
place fails.

In [ ]:
lfp *= uV_per_count

The power of a signal is its square. Squaring the raw counts goes wrong:

In [ ]:
(lfp**2).min()  # a square can't be negative

1. Why does the in-place conversion fail, and what is the dtype of `lfp_uv`?
2. Why is the square negative? (Hint: what is 225 squared, and what is the
   largest `int16`?)
3. Make `lfp_power`, the square of every sample in microvolts squared, and
   check that its minimum is not negative.

In [ ]:
# Your code here


*Your answer here.*

### 5e. More broadcasting

1. The sites of a Neuropixels probe sit in a staggered pattern: `x_um` and
   `depth_um` give each site's position in micrometres. Make `distances`, the
   384 x 384 array where `distances[i, j]` is the distance between site `i`
   and site `j`. (Hint: stack the positions into `positions`, shape (384, 2).
   Give one copy the shape (384, 1, 2) and another (1, 384, 2). Their
   difference has shape (384, 384, 2). Then square, sum over the last axis
   and take the square root.) Check that the diagonal is 0, and look at
   `distances[:4, :4]`.

In [ ]:
# Your code here


2. **(SPL Exercise 22, part 2)** Divide each **column** of `a` element-wise
   by `b`, so that row `i` is divided by `b[i]`. (Hint: `np.newaxis`.)

In [ ]:
a = np.arange(25).reshape(5, 5)
b = np.array([1.0, 5, 10, 15, 20])

In [ ]:
# Your code here


### 5f. More masks

1. `layer_5_somatosensory`: the LFP of the sites in layer 5 (`layer == "5"`)
   of **either** somatosensory area, `"SSp-n"` or `"SSs"`. How many sites is
   that? (Hint: `np.isin`, or two comparisons joined with `|`.)
2. `silent_trials`: for unit 0, the number of trials in which it fired **no
   spikes at all**. (Hint: `np.all` with an `axis`.)

In [ ]:
# Your code here


### 5g. Fancy indexing

An integer array can index into a **lookup table**: every index is replaced
by the matching entry.

1. `unit_channel` gives the channel on which each of the 481 neurons (units)
   was largest. Make `unit_area`: the brain area of each unit, using `area`
   as the lookup table. How many units are in `"SSp-n"`? How many are
   in `"void"` sites, above the brain?

In [ ]:
# Your code here


2. Make `rt_sorted`, the response times sorted fastest first, and
   `correct_sorted`, the `correct` values reordered **the same way** so that
   they still belong to the same trials. (Hint: `argsort`.)

In [ ]:
# Your code here


3. To draw the probe, we want one colour per channel, by area. `area_names`
   lists the areas and `area_code` gives each channel's position in that list
   (so `area_names[area_code]` is `area` again). `palette` gives one RGB
   colour per area. Make `channel_colours`, an RGB colour for every channel,
   using **one** indexing expression. What shape is it, and why?

Then, in place:

4. Channels 12, 57 and 300 are known to be noisy. Set them to 0 in
   `lfp_uv`, using one integer list as the index.

In [ ]:
area_names, area_code = np.unique(area, return_inverse=True)
palette = rng.integers(0, 256, size=(area_names.size, 3))
area_names, area_code[:12]

In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


5. **(SPL Exercise 22, part 3; harder)** `r` is a 10 x 3 array of random
   numbers in [0, 1]. Make `closest`: for each row, the number closest to
   0.5.

   * Use `abs` and `argmin` to find the column `j` closest to 0.5 in each row.
   * Use fancy indexing to extract the numbers. (Hint: `r[i, j]`, where the
     array `i` contains the row numbers that go with `j`.)

In [ ]:
r = rng.random((10, 3))

In [ ]:
# Your code here


### 5h. Data statistics (after SPL Exercise 23)

Time to put it together. `spike_unit` gives the unit number of each row of
`spike_counts`, the same numbering as in `unit_channel`.

In [ ]:
with h5py.File("../data/ibl_session.h5") as f:
    spike_unit = f["spike_counts/unit"][:]
spike_unit[:10]

Compute and print, **without any for-loops**:

1. The mean and standard deviation of the response time, for correct and for
   wrong trials. What does the difference tell you about the mouse?
2. For each unit, the time (from `bin_start`) of the bin where its
   trial-averaged count is highest. (Hint: `argmax` of `psth`, then fancy
   indexing of `bin_start`.)
3. For each trial, the **unit number** that fired the most spikes over the
   whole trial. Is it the same unit every time? (Hint: `argmax` and fancy
   indexing of `spike_unit`.)
4. The trials in which **any** unit fired more than 10 spikes in a single
   50 ms bin. (Hint: comparisons, and `np.any` over two axes.)
5. The unit numbers of the two units that fired the most spikes in total.
   (Hint: `argsort` and fancy indexing.)

(Part 6, the population response on correct and wrong trials, is a stretch
exercise in Day 2's Matplotlib notebook.)

In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


In [ ]:
# Your code here
